# Project 13 — SnelBite Food Delivery: Gold Layer Practice

**Format:** Gold-layer drill (no cleaning this time) · **Time-box: 60–90 min**

## Scenario

You are a data engineer at **SnelBite**, a food delivery platform operating in 5 Dutch cities.
The ingestion team has already cleaned the data: `food_delivery_13.csv` is the **silver table**
(no duplicates, no nulls, consistent formats). Your job is the **gold layer**: answer the
10 business questions below with Spark, and write the final gold tables.

## Data dictionary

| column | type (target) | notes |
|---|---|---|
| order_id | string | unique per order |
| order_date | date | ISO format, 2025-01-01 → 2025-06-30 |
| city | string | 5 cities |
| restaurant_name | string | 24 restaurants |
| cuisine | string | 6 cuisines |
| customer_id | string | C-xxx |
| order_amount | decimal(10,2) | EUR |
| delivery_minutes | int | |
| rating | int | 1–5 |

## Rules

- Upload the CSV to Databricks yourself and read it (schema is your call — data is clean, but casting `order_date` to date and `order_amount` to decimal is still on you).
- Every monthly result must use the **yyyy-MM** format (you know why).
- Questions marked **[W]** must be solved with a **window function**.
- Q10 must be written as a table (idempotent overwrite).


## Setup — read the data

Read the CSV, cast the columns to their target types, and run a quick sanity check
(row count, 0 duplicates, 0 nulls) so you trust the silver input.

In [ ]:
food_delivery_gold_df = (
    spark.read
    .format("csv")
    .option("header",True)
    .load("/Volumes/dev/spark_db/datasets/mini-projects/raw_data/food_delivery_13.csv")
)

food_delivery_gold_df.display()

In [ ]:
from pyspark.sql.functions import isnull, col

food_delivery_gold_df.filter(
    col("order_id").isNull() | col("order_date").isNull() | col("city").isNull() | col("restaurant_name").isNull() |
    col("cuisine").isNull()  | col("customer_id").isNull() | col("order_amount").isNull() | col("delivery_minutes").isNull()
    | col("rating").isNull()
).display()

In [ ]:
print(food_delivery_gold_df.dropDuplicates().count())
print(food_delivery_gold_df.count())

# NO DUPLICATE VALUES

In [ ]:
food_delivery_gold_df.printSchema()

In [ ]:
food_delivery_gold_df = (
    food_delivery_gold_df.withColumns({
        "order_date" : col("order_date").cast("date"),
        "order_amount" : col("order_amount").cast("decimal(10,2)"),
        "delivery_minutes" : col("delivery_minutes").cast("int"),
        "rating" : col("rating").cast("int")
    })
)

food_delivery_gold_df.printSchema()

In [ ]:
food_delivery_gold_df.filter(col("order_date").isNull()).count()

## Q1 — Revenue per city

Total revenue and number of orders **per city**, sorted by revenue descending. Which city generates the most revenue?

In [ ]:
from pyspark.sql.functions import col, expr, sum, round, count
revenue_per_city_df = (
    food_delivery_gold_df.groupBy("city").agg(
        round(sum(col("order_amount")),2).alias("total_revenue"),
        count(col("order_id")).alias("number_of_order")
)
).orderBy(col("total_revenue").desc())

revenue_per_city_df.display()


In [ ]:
# GRONINGEN GENERATES THE MOST REVENUE IN THE NETHERLANDS.

## Q2 — Average order value per cuisine

Average `order_amount` **per cuisine**, rounded to 2 decimals, sorted descending. Which cuisine has the highest average ticket?

In [ ]:
from pyspark.sql.functions import avg

avg_value_per_cuisine = (
    food_delivery_gold_df.groupBy("cuisine").agg(
        round(avg("order_amount"),2).alias("avg_order_value")
    )
).orderBy(col("avg_order_value").desc())

avg_value_per_cuisine.display()

In [ ]:
# JAPANESE CUISINE GENERATES THE HIGHEST AVERAGE TICKET IN THE NETHERLANDS.

## Q3 — Monthly revenue

Total revenue **per month** (yyyy-MM). Which month was the strongest, which was the weakest?

In [ ]:
from pyspark.sql.functions import date_format

monthly_revenue_df = food_delivery_gold_df.withColumn("order_month", date_format("order_date", "yyyy-MM")).groupBy("order_month").agg(
        sum("order_amount").alias("monthly_revenue")
    ).orderBy(col("monthly_revenue"))

monthly_revenue_df.display()
                              

In [ ]:
#THE STRONGEST MONTH WAS MARCH, THE WEAKEST MONTH WAS APRIL.

## Q4 — Delivery speed per city

Average `delivery_minutes` and order count **per city**. Which city is the slowest? Would you trust that number the same for every city — why / why not?

In [ ]:
deliver_speed_per_city_df = (
    food_delivery_gold_df.groupBy("city").agg(
        round(avg("delivery_minutes"),2).alias("avg_delivery_minutes"),
        count("order_id").alias("order_count"))
).orderBy(col("avg_delivery_minutes"))

deliver_speed_per_city_df.display()

In [ ]:
# THE SLOWEST CITY IS GRONINGEN, THE FASTEST CITY IS EINDHOVEN.GRONINGEN IS 46% SLOWER THAN EINDHOVEN ( 37.51 / 25.7 = 1.46 = 46% ).TO TRUST AVG VALUES WE ALSO SHOULD CALCULATE ORDER COUNT BECAUSE IF THERE IS JUST 3 ORDERS IT WOULD BE DECEPTIVE FOR US.IN THIS CASE ALL FIVE CITIES HAVE ENOUGH COUNT OF OBSERVATIONS SO WE CAN TRUST THEM. AS THE COUNT OF OBSERVATIONS BIGGER, IT GETS MORE RELIABLE.

## Q5 — Loyal customers

How many customers placed **5 or more orders**? Show the top 3 customers by order count.

In [ ]:
loyal_customers_df = food_delivery_gold_df.groupBy("customer_id").count().filter(col("count")>=5).orderBy(col("count").desc())

loyal_customers_df.limit(3).display()

print(loyal_customers_df.count())

In [ ]:
# THERE ARE 43 CUSTOMERS WHO ORDERS 5 TIMES OR MORE THAN 5 TIMES. 

## Q6 [W] — Top 3 restaurants per city

For **each city**, find the **top 3 restaurants by total revenue**. Result: city, restaurant_name, revenue, rank (1–3).

In [ ]:
food_delivery_gold_df.createOrReplaceTempView("food_delivery_gold")

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, row_number

restaurant_reveue_df = food_delivery_gold_df.groupBy("city","restaurant_name").agg(
    sum("order_amount").alias("total_revenue")
)

w = Window.partitionBy("city").orderBy(col("total_revenue").desc())

top_3_restaurant_per_city_df = restaurant_reveue_df.withColumn("rn", row_number().over(w)).filter(col("rn")<=3)

top_3_restaurant_per_city_df.display()



In [ ]:
%sql

with x as (
select restaurant_name,
city,
sum(order_amount) as total_revenue
from food_delivery_gold 
group by city, restaurant_name 
),

y as (
select restaurant_name, city, total_revenue,
row_number() over(partition by city order by total_revenue desc ) as rn
from x 
)

select city, restaurant_name, total_revenue, rn from y 
where rn<= 3





## Q7 [W] — Month-over-month change per city

For **each city**, compute monthly revenue and the **change vs the previous month**. Which city had the single biggest monthly drop, and in which month?

In [ ]:
from pyspark.sql.functions import date_format, expr,col, lag
from pyspark.sql.window import Window

monthly_df = food_delivery_gold_df.groupBy("city",date_format("order_date", "yyyy-MM").alias("order_month")).agg(
    sum("order_amount").alias("total_revenue")
)

w = ( Window.partitionBy("city")
     .orderBy("order_month")
     )

monthly_diff_df = monthly_df.withColumn("previous_month_revenue", lag(col("total_revenue")).over(w))

result_df = monthly_diff_df.withColumns({
    "revenue_diff" : expr("total_revenue - previous_month_revenue")
}).orderBy(col("revenue_diff").asc_nulls_last())

result_df.display()



In [ ]:
%sql


with x as (
select city,
date_format(order_date, "yyyy-MM") as order_month,
sum(order_amount) as total_revenue from food_delivery_gold
group by city, date_format(order_date, "yyyy-MM")
),

y as (
select city, order_month, total_revenue,
lag(total_revenue) over(partition by city order by order_month) as previous_month_revenue
from x 
)

select city,
order_month,
total_revenue,
previous_month_revenue,
(total_revenue - previous_month_revenue) as revenue_diff
from y
order by revenue_diff nulls last


-- GRONNINGEN HAS THE BIGGEST MONTHLY DROP ON APRIL.









In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, sum

w = Window.orderBy("order_month")

food_delivery_month_df = food_delivery_gold_df.withColumns({
    "order_month"    : date_format("order_date", "yyyy-MM").alias("order_month")
})

order_month_df = food_delivery_month_df.groupBy("order_month").agg(
    sum(col("order_amount")).alias("total_revenue")
)

result_df = order_month_df.withColumn("cumulative_revenue_total", sum(col("total_revenue")).over(w)).where(col("cumulative_revenue_total")>15000)

result_df.display()




## Q8 [W] — Running total

Compute the **cumulative (running) revenue by month** for the whole company. In which month did cumulative revenue pass **€15,000**?

In [ ]:
%sql

with x as (
select
date_format(order_date, "yyyy-MM") as order_month,
sum(order_amount) as total_revenue
from food_delivery_gold
group by order_month
order by order_month
),

y as (
select 
order_month,
total_revenue,
sum(total_revenue) over(order by  order_month) as cumulative_revenue
from x 
)

select order_month,
total_revenue,
cumulative_revenue
from y 
where cumulative_revenue > 15000







## Q9 [W] — Order vs city average

For each order, add a column with the **average order_amount of its city** (without losing the order-level rows) and a column `diff_vs_city_avg`. How many orders are **above** their city's average? Then answer: why can't a plain `groupBy` alone produce this result?

In [ ]:
%sql

with x as (
select order_id, 
city,
order_amount,
avg(order_amount) over(partition by city) as avg_revenue
from food_delivery_gold
)

select * from x 
where order_amount > avg_revenue

-- WE CANNOT USE GROUPBY HERE BECAUSE IF WE USE IT WE WOULD LOST ORDER-LEVEL ROWS. 








## Q10 — Gold table: city × month

Build the gold table `gold_city_monthly` with: city, month (yyyy-MM), total revenue, order count, avg delivery minutes. Write it as a table with an **idempotent overwrite**. Which city+month combination has the highest revenue?

In [ ]:
from pyspark.sql.functions import count,sum,avg

result_df = food_delivery_gold_df.withColumns({
    "month"  : date_format(col("order_date"),"yyyy-MM")
}).groupBy("city","month").agg(
    sum("order_amount").alias("total_revenue"),
    count("order_id").alias("nb_of_order"),
    avg("delivery_minutes").alias("avg_delivery_minutes")
)

result_df.write.mode("overwrite").saveAsTable("gold_city_monthly")

In [ ]:
spark.read.table("gold_city_monthly").display()

In [ ]:
spark.table("gold_city_monthly").orderBy(col("total_revenue").desc()).limit(1).display()

# HIGHEST REVENUE COMBINATION: GRONINGEN, 2025-03 (1,491.09 EUR)

## BONUS — Q11: Unit test your Q6 logic (guided)

This part is **guided** — follow the steps. You are going to unit test the "top N restaurants
per city" logic from Q6. The finished module and test file live next to this notebook in the repo:
`gold_functions.py` and `test_gold_functions.py`.

### Step 1 — Extract the logic into a module

Create a file **`gold_functions.py`** in the same folder as this notebook, with ONE function:

- name: `top_n_per_city(df, n=3)`
- input: a DataFrame with columns `city`, `restaurant_name`, `revenue` (already aggregated — one row per city+restaurant)
- output: top n restaurants per city by revenue
- rules: the function must **return** a DataFrame, must NOT read any file, must NOT create a SparkSession
- ⚠️ use **`row_number()`**, and order by `revenue` **descending, then `restaurant_name` ascending** as a tie-breaker (you will see why in Step 3)

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col

w = Window.partitionBy("city").orderBy(col("total_revenue").desc(),col("restaurant_name").asc())

food_delivery_new_df = food_delivery_gold_df.groupBy("restaurant_name","city").agg(
    sum("order_amount").alias("total_revenue")
)

top_3_df = (
    food_delivery_new_df.withColumn("rn", row_number().over(w))
    .filter(col("rn")<=3).drop("rn")
)

top_3_df.display()







### Step 2 — Create the test file

Create **`test_gold_functions.py`** next to it. Start from the standard skeleton (imports + `spark` fixture). The only line that changes
from project to project is the import:
`from gold_functions import top_n_per_city`

### Step 3 — Build the mini test data BY HAND

In your test function, build this exact input with `spark.createDataFrame`
(schema: `"city string, restaurant_name string, revenue double"`):

| city | restaurant_name | revenue |
|---|---|---|
| A | R1 | 400.0 |
| A | R2 | 300.0 |
| A | R3 | 200.0 |
| A | R4 | 200.0 |
| A | R5 | 100.0 |
| B | R6 | 80.0 |
| B | R7 | 60.0 |

Two traps are built in on purpose:
- city A has a **tie** at 200.0 (R3 vs R4) — this is why `rank()` + `where("rank<=3")` would return **4 rows**, and why `row_number()` + a tie-breaker gives a deterministic 3
- city B has **fewer rows than n** — the function must return 2 rows for B, not fail

Now write **`expected_df` by hand** (5 rows total — work out which ones yourself, on paper).
Do NOT produce expected_df by running your own function — that would prove nothing (circular proof,
you know this one from Project 12).

In [ ]:
import pytest
import sys

sys.dont_write_bytecode = True
retcode = pytest.main(["-v", "test_gold_functions.py"])
assert retcode == 0, "Tests failed!"

### Step 5 — Answer briefly (in English, for the review)

1. Why did we test with 7 hand-written rows instead of the real 850-row CSV?
2. What would happen to your test if you had used `rank()` instead of `row_number()`?

**1. Why did we test with 7 hand-written rows instead of the real 850-row CSV?**

Because with 7 hand-written rows I know the correct answer in advance, so the
test actually proves the logic. If I used the real 850-row CSV, I would have to
run my own function to produce the "expected" output — the code would be
checking itself. That is circular proof, and it proves nothing.

**2. What would happen to your test if you had used rank() instead of row_number()?**

The test would fail. City A has a tie at 200.0 (R3 and R4), and rank() gives
tied rows the same rank, so the filter rank <= 3 would return 4 rows for city A
instead of 3. assertDataFrameEqual would catch the extra row. row_number() plus
a tie-breaker (restaurant_name asc) guarantees a deterministic top 3.

## Defense questions (answered in English)

1. Q6: you aggregated first, then applied the window. Why does that order matter — what breaks if you apply `row_number` on the raw order-level rows?
2. Q7: what does `lag()` return for the first month of each city, and how does your result handle it?
3. Q9: explain the difference between a `groupBy` aggregate and a window aggregate (`avg over partition`) in terms of the grain of the output.
4. Your Q10 table will be rebuilt every day by a scheduled job. Why is overwrite the right write mode here, and when would it be the wrong choice?
5. Q11: your two tests passed. What do they prove — and what do they NOT prove about your pipeline? (hint: unit test vs data quality test)
6. Q12: your rule also works with `when/otherwise`. Why would production code prefer the built-in over your UDF — and what exactly does `useArrow=True` make faster?


1. (Q6 — why aggregate first?)
row_number on raw order rows would rank individual orders, not restaurants.
I first aggregate to one row per city+restaurant, so the window ranks
restaurants by their total revenue. Wrong grain in = wrong ranking out.

2. (Q7 — lag on the first month?)
lag() returns null for the first month, because there is no previous row
inside the partition. That null is information, not an error — it means
"no previous month". I kept it and sorted with nulls last, so it does not
pollute the "biggest drop" ranking.

3. (Q9 — groupBy vs window aggregate, grain)
A groupBy aggregate collapses the grain: 850 orders become 5 city rows.
A window aggregate keeps the grain: all 850 rows stay, and the city average
is broadcast onto each row. Same math, different output grain.

4. (Q10 — why overwrite?)
The job rebuilds the WHOLE table from source every day, so overwrite makes
the run idempotent: running it twice gives the same result. Append would
duplicate all rows on every rerun. Overwrite would be wrong for incremental
loads — if I only process today's data, overwrite would delete history.

5. (Q11 — what do the tests prove / not prove?)
They prove the LOGIC is correct: top-n selection, the tie-breaker, and the
small-group case behave as designed. They do NOT prove the real data is
correct — nulls, duplicates or bad values in the 850-row table need data
quality checks, which run on real data, not hand-made rows.

6. (Q12 — built-in vs UDF, useArrow?)
Built-ins like when/otherwise run inside Spark's engine, so they are
optimized and faster. A Python UDF forces Spark to move every row out to a
Python process and back — slow serialization. useArrow=True makes that
transfer faster by batching rows in Arrow format instead of one-by-one
pickling — but it only softens the cost; the built-in is still better.

## Key Takeaways

- **Aggregate first, then window.** Ranking restaurants means the window must run on
  city+restaurant grain, not on raw order rows. Wrong grain in = wrong ranking out.
- **`partition by` vs `order by` inside a window:** partition = where the counter resets,
  order = the sequence the function walks. `lag`/running totals need `order by`;
  "broadcast the group value onto every row" (Q9) must NOT have one — adding it silently
  turns a group average into a running average.
- **A window with `order by` defaults to a running frame** (`range unbounded preceding →
  current row`). Write the frame explicitly when you mean a running total.
- **`groupBy` collapses grain; a window keeps it.** That is the whole answer to
  "why can't a plain groupBy produce a per-order comparison column".
- **Idempotent overwrite paid off immediately:** the gold table spec changed mid-exercise
  (avg delivery minutes, not avg order value) — rerunning the overwrite write fixed the
  table with no duplicate rows.
- **First unit-tested project:** the Q6 logic moved into `gold_functions.py` and got two
  pytest tests with 7 hand-written rows — a deliberate tie (row_number + tie-breaker vs
  rank) and a group smaller than n. Expected output written by hand, never generated by
  the function itself (circular proof). The `assert retcode == 0` line turns test results
  into a pipeline gate.